# Gridded COARDS NetCDF files

COARDS-convention rectilinear grids (packed int16 with scale/offset). Same workflow as the CF notebook: inspect, plot, retrieve, reduce, mutate, crop, save.

In [ ]:
%matplotlib inline
from pathlib import Path
import tempfile

import numpy as np
import geopandas as gpd
from shapely.geometry import Polygon

from pyramids.dataset import Dataset
from pyramids.netcdf import NetCDF, UgridDataset
from pyramids.feature import FeatureCollection

DATA = Path('../../../../examples/data/netcdf/samples')

## `coards__4v__1d3-3d1.nc`

NCEP reanalysis air temperature on a 2.5° global grid (packed int16).

**Open the file and inspect the container**

In [ ]:
nc = NetCDF.read_file(DATA / 'coards__4v__1d3-3d1.nc')
nc

**Dimensions and variables**

In [ ]:
# get_all_metadata() returns a NetCDFMetadata whose summary lists dimensions and, for each
# variable, its dims / shape / dtype / unit / scale-offset.
meta = nc.get_all_metadata()
print(meta)

**Global attributes**

In [ ]:
nc.global_attributes

**Plot the variable** (a 2-D slice is auto-selected for >2-D variables)

In [ ]:
# this COARDS file has no embedded CRS and uses 0..360 longitudes
air0 = nc.get_variable('air')
# read the data and take the first time step (a 2-D lat/lon slice)
air_data = air0.read_array()
air_slice = air_data[0]
# shift the geotransform origin by -360 so the longitudes become -180..180
gt = air0.geotransform
shifted_gt = (gt[0] - 360, gt[1], gt[2], gt[3], gt[4], gt[5])
# rebuild the slice as a georeferenced Dataset (assign EPSG:4326, matching its lon/lat axes)
air_geo = Dataset.create_from_array(air_slice, geo=shifted_gt, epsg=4326)
# plot over an OpenStreetMap basemap
glyph = air_geo.plot()
# draw the Natural Earth coastline on top so the geography stays visible over the data
glyph.add_features("coastline", "50m", crs=4326, zorder=5)

**Retrieve the underlying data**

In [ ]:
var = nc.get_variable('air')
data = var.read_array()
print('shape:', data.shape)
print('min / mean / max:', float(np.nanmin(data)), float(np.nanmean(data)), float(np.nanmax(data)))

**Reduce a dimension** — collapse the time axis to its mean

In [ ]:
time_mean = nc.reduce('time', how='mean')
print('dimensions after reducing time:', dict(time_mean.dimension_sizes))

**Add a variable** — derive a 2-D field and append it as a new variable

In [ ]:
work = Path(tempfile.mkdtemp())
slice2d = data[tuple(0 for _ in range(data.ndim - 2))]
NetCDF.create_from_array(
    arr=slice2d, geo=var.geotransform, epsg=var.epsg or 4326,
    variable_name='air_slice0', path=str(work / 'derived.nc'),
)
nc.add_variable(NetCDF.read_file(str(work / 'derived.nc')), 'air_slice0')
print('variables after add:', nc.variable_names)

**Remove a variable**

In [ ]:
nc.remove_variable('air_slice0')
print('variables after remove:', nc.variable_names)

**Crop with a real-world polygon** — this COARDS file ships **without an embedded CRS** and uses 0–360 longitudes, so we rebuild one time step as a georeferenced `Dataset` shifted to −180…180 (assigning `EPSG:4326`). That lets us crop to the **contiguous United States** *and* line the result up with the **OpenStreetMap basemap** (we reproject the crop to Web Mercator / EPSG:3857; tiles are fetched over the network) — coastline, Gulf and Florida all visible.

In [ ]:
# Contiguous USA, in -180..180 longitudes
usa = Polygon([
    (-125, 49), (-102, 49), (-80, 47), (-66, 45), (-66, 38), (-77, 30),
    (-80, 25), (-90, 30), (-98, 30), (-105, 25), (-112, 33), (-122, 42), (-125, 49),
])
aoi = FeatureCollection(gpd.GeoDataFrame(geometry=[usa], crs=4326))

# the file has no embedded CRS + uses 0..360 lons, so shift the geotransform origin by -360
gt = var.geotransform
shifted_gt = (gt[0] - 360, gt[1], gt[2], gt[3], gt[4], gt[5])
# rebuild the first time step as a georeferenced Dataset at -180..180 (EPSG:4326)
air_slice = data[0]
air_geo = Dataset.create_from_array(air_slice, geo=shifted_gt, epsg=4326)
# crop to the USA polygon
cropped = air_geo.crop(aoi)
print('cropped air bounds:', [round(b, 1) for b in cropped.total_bounds])
glyph = cropped.plot()
# draw the Natural Earth coastline on top so the geography stays visible over the data
glyph.add_features("coastline", "50m", crs=4326, zorder=5)

**Save the result** — the cropped field is a single-band raster, written as a GeoTIFF.

In [ ]:
out = work / 'air_usa.tif'
cropped.to_file(out)
print('saved cropped raster to', out.name, '->', out.exists())

## `coards__5v__1d4-4d1.nc`

Relative humidity with a vertical level dimension (4-D, packed int16). Longitudes run 0–360.

**Open the file and inspect the container**

In [ ]:
nc = NetCDF.read_file(DATA / 'coards__5v__1d4-4d1.nc')
nc

**Dimensions and variables**

In [ ]:
# get_all_metadata() returns a NetCDFMetadata whose summary lists dimensions and, for each
# variable, its dims / shape / dtype / unit / scale-offset.
meta = nc.get_all_metadata()
print(meta)

**Global attributes**

In [ ]:
nc.global_attributes

**Plot the variable** (a 2-D slice is auto-selected for >2-D variables)

In [ ]:
# select the variable to plot
field = nc.get_variable('rhum')
# this grid's longitudes run 0..360; wrap them to -180..180 so it lines up with the coastline
field = field.wrap_longitude()
# plot in the data's own CRS (lon/lat degrees) — no reprojection
glyph = field.plot()
# draw the Natural Earth coastline on top so the geography stays visible over the data
glyph.add_features("coastline", "50m", crs=4326, zorder=5)

**Wrap longitude to −180–180** — this grid spans 0–360 (Pacific-centred). `wrap_longitude` re-frames it to the −180–180 (Greenwich-centred) convention; we wrap the selected variable, then plot the re-centred map.

In [ ]:
# select the variable to plot
field = nc.get_variable('rhum')
# this grid's longitudes run 0..360; wrap them to -180..180 so it lines up with the coastline
field = field.wrap_longitude()
# plot in the data's own CRS (lon/lat degrees) — no reprojection
glyph = field.plot()
# draw the Natural Earth coastline on top so the geography stays visible over the data
glyph.add_features("coastline", "50m", crs=4326, zorder=5)

**Retrieve the underlying data**

In [ ]:
var = nc.get_variable('rhum')
data = var.read_array()
print('shape:', data.shape)
print('min / mean / max:', float(np.nanmin(data)), float(np.nanmean(data)), float(np.nanmax(data)))

**Reduce a dimension** — collapse the time axis to its mean

In [ ]:
time_mean = nc.reduce('time', how='mean')
print('dimensions after reducing time:', dict(time_mean.dimension_sizes))

**Add a variable** — derive a 2-D field and append it as a new variable

In [ ]:
work = Path(tempfile.mkdtemp())
slice2d = data[tuple(0 for _ in range(data.ndim - 2))]
NetCDF.create_from_array(
    arr=slice2d, geo=var.geotransform, epsg=var.epsg or 4326,
    variable_name='rhum_slice0', path=str(work / 'derived.nc'),
)
nc.add_variable(NetCDF.read_file(str(work / 'derived.nc')), 'rhum_slice0')
print('variables after add:', nc.variable_names)

**Remove a variable**

In [ ]:
nc.remove_variable('rhum_slice0')
print('variables after remove:', nc.variable_names)

**Crop with a real-world polygon** — this grid is global and coarse (5°), so we crop to a compact, recognisable continent: **Australia**. `crop` masks every cell outside the polygon to no-data and trims to its bounds; at 5° the continent is blocky but unmistakable. We crop one variable (shown over the basemap, reprojected to Web Mercator / EPSG:3857), then the whole container in native 0–360 coordinates (no basemap). The result is drawn over an **OpenStreetMap basemap** (`basemap=True`) for geographic context — note this fetches map tiles over the network at run time.

In [ ]:
# Australia, lon in the file's native 0-360 convention
australia = Polygon([
    (113, -22), (123, -17), (130, -12), (138, -12), (143, -11), (147, -19),
    (153, -28), (150, -37), (141, -38), (135, -35), (129, -32), (123, -34),
    (115, -34), (113, -26), (113, -22),
])
aoi = FeatureCollection(gpd.GeoDataFrame(geometry=[australia], crs=var.epsg or 4326))

# select the variable
rhum = nc.get_variable('rhum')
# crop to the Australia polygon (no wrap needed: 113..153 is already valid in -180..180)
rhum_au = rhum.crop(aoi)
print('cropped rhum bounds:', [round(b, 1) for b in rhum_au.total_bounds])
glyph = rhum_au.plot()
# draw the Natural Earth coastline on top so the geography stays visible over the data
glyph.add_features("coastline", "50m", crs=4326, zorder=5)

**Crop the whole container** — clip every variable with the same polygon.

In [ ]:
# crop every variable in the container at once (the Australia polygon is valid in 0-360 too)
cropped = nc.crop(aoi)
print('variables in cropped container:', cropped.variable_names)
glyph = rhum_au.plot()
# draw the Natural Earth coastline on top so the geography stays visible over the data
glyph.add_features("coastline", "50m", crs=4326, zorder=5)

**Save the result to a new NetCDF file**

In [ ]:
out = work / 'cropped.nc'
cropped.to_file(out)
print('saved cropped container to', out.name, '->', out.exists())